<a href="https://colab.research.google.com/github/huseyincenik/john_snow_labs/blob/main/generating_conll_files_from_pretrained_models/notebooks/data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preparation - NER Pipeline and CoNLL Generation

This notebook loads healthcare datasets, runs NER pipeline, extracts entities, and saves them in CoNLL format.

**Google Drive Integration:**
- All files are saved to Google Drive
- Files are read from Google Drive
- All code is embedded in this notebook (no external Python files required)

## Steps:
1. **Google Drive Connection** - Mount Google Drive
2. **Setup & License** - Spark NLP Healthcare license and environment setup
3. **Dataset Loading** - Load and prepare healthcare datasets
4. **NER Pipeline** - Run NER pipeline with pre-trained models
5. **Entity Extraction** - Extract and merge entities (Priority: Posology > DeID > Clinical)
6. **CoNLL Generation** - Convert entities to CoNLL format

**Outputs:**
- `data/processed/text_data.csv` - Prepared text data
- `data/processed/entities.csv` - Extracted entities
- `data/conll/conll2003_text_file.conll` - CoNLL format training data


## 1. Google Drive Connection


In [1]:
# Mount Google Drive
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Set project folder in Google Drive
PROJECT_FOLDER = '/content/drive/MyDrive/john_snow_labs_ner'
os.makedirs(PROJECT_FOLDER, exist_ok=True)

# Change working directory
os.chdir(PROJECT_FOLDER)

# Create folder structure
for folder in ['data/raw', 'data/processed', 'data/conll', 'cache_pretrained']:
    os.makedirs(folder, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"✅ Project folder: {PROJECT_FOLDER}")
print(f"✅ Current directory: {os.getcwd()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted
✅ Project folder: /content/drive/MyDrive/john_snow_labs_ner
✅ Current directory: /content/drive/MyDrive/john_snow_labs_ner


## 2. Setup & License Configuration


In [2]:
import json
import os

# Load license keys from Google Drive
license_path = f'{PROJECT_FOLDER}/spark_jsl.json'
if not os.path.exists(license_path):
    print("❌ License file not found!")
    print("Please upload spark_jsl.json to Google Drive at the project folder")
    print("You can upload it manually or use the following code:")
    print("from google.colab import files")
    print("uploaded = files.upload()")
    raise FileNotFoundError(f"License file not found at {license_path}")

with open(license_path) as f:
    license_keys = json.load(f)

# Set license keys as environment variables
locals().update(license_keys)
os.environ.update(license_keys)

print("✅ License keys loaded")
print(f"JSL Version: {license_keys.get('JSL_VERSION', 'N/A')}")
print(f"Public Version: {license_keys.get('PUBLIC_VERSION', 'N/A')}")


✅ License keys loaded
JSL Version: 6.2.1
Public Version: 6.2.0


In [3]:
# Install Java (required for Spark)

import subprocess
import os

try:
    java_version = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT, text=True)
    print(f"✅ Java is already installed: {java_version.split(chr(10))[0]}")

    if 'JAVA_HOME' not in os.environ:
        java_paths = [
            "/usr/lib/jvm/java-11-openjdk-amd64",
            "/usr/lib/jvm/java-8-openjdk-amd64",
            "/usr/lib/jvm/default-java"
        ]
        for path in java_paths:
            if os.path.exists(path):
                os.environ["JAVA_HOME"] = path
                print(f"✅ Set JAVA_HOME to: {path}")
                break
except Exception as e:
    print(f"Java check failed: {e}")
    print("Installing Java 11...")
    os.system('apt-get update -qq > /dev/null 2>&1')
    os.system('apt-get -y install -qq openjdk-11-jdk > /dev/null 2>&1')
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    print("✅ Java 11 installation attempted")


# Check GPU availability safely
try:
    gpu_check = subprocess.run(
        ['nvidia-smi'],
        capture_output=True,
        text=True
    )
    has_gpu = gpu_check.returncode == 0
except FileNotFoundError:
    has_gpu = False

if has_gpu:
    print("🚀 GPU detected! Installing PyTorch with CUDA support...")
    %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
else:
    print("⚠️ GPU not detected. Installing PyTorch CPU version...")
    %pip install -q torch torchvision torchaudio


# Install PySpark and Spark NLP
%pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION

# Install Spark NLP Healthcare
%pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Install additional dependencies
%pip install -q pandas numpy tqdm requests

print("✅ All libraries installed successfully!")
if has_gpu:
    print("✅ GPU-accelerated PyTorch installed")


✅ Java is already installed: openjdk version "17.0.16" 2025-07-15
⚠️ GPU not detected. Installing PyTorch CPU version...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spark-nlp-jsl 6.2.1 requires spark-nlp==6.2.2, but you have spark-nlp 6.2.0 which is incompatible.
✅ All libraries installed successfully!


In [4]:
import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from sparknlp_jsl.annotator import MedicalNerModel, NerConverter
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        print(f"🚀 GPU Detected: {gpu_name}")
    else:
        print("⚠️  No GPU detected. Using CPU mode.")
except ImportError:
    print("⚠️  PyTorch not available. GPU check skipped.")
    gpu_available = False

# Spark configuration
params = {
    "spark.driver.memory": "8G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize": "2000M",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer"
}

if gpu_available:
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.storage.cluster_tmp_dir": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.annotator.gpu": "true"
    })
    print("✅ GPU acceleration enabled in Spark configuration")

# Start Spark session
try:
    print("Starting Spark session...")
    spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
    spark.sparkContext.setLogLevel("ERROR")

    print(f"✅ Spark NLP Version: {sparknlp.version()}")
    print(f"✅ Spark NLP JSL Version: {sparknlp_jsl.version()}")
    print("✅ Spark session initialized successfully")

except Exception as e:
    print(f"❌ Error starting Spark session: {e}")
    raise

spark


⚠️  No GPU detected. Using CPU mode.
Starting Spark session...
✅ Spark NLP Version: 6.2.2
✅ Spark NLP JSL Version: 6.2.1
✅ Spark session initialized successfully


## 3. Dataset Loading


In [5]:
# Download dataset
import requests

data_dir = Path(f"{PROJECT_FOLDER}/data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

url = "https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp-workshop/master/tutorials/Certification_Trainings/Healthcare/data/mtsamples_classifier.csv"
file_path = data_dir / "mtsamples_classifier.csv"

if not file_path.exists():
    print(f"Downloading mtsamples_classifier dataset from {url}...")
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)
    print(f"Dataset saved to {file_path}")
else:
    print(f"Dataset already exists at {file_path}")

df = pd.read_csv(file_path)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


Dataset saved to /content/drive/MyDrive/john_snow_labs_ner/data/raw/mtsamples_classifier.csv
Dataset shape: (638, 2)
Columns: ['category', 'text']


,category,text
0,Gastroenterology,PROCEDURES PERFORMED: Colonoscopy. INDICATION...
1,Gastroenterology,OPERATION 1. Ivor-Lewis esophagogastrectomy. ...
2,Gastroenterology,PREOPERATIVE DIAGNOSES: 1. Gastroesophageal r...
3,Gastroenterology,PROCEDURE: Colonoscopy. PREOPERATIVE DIAGNOSE...
4,Gastroenterology,PREOPERATIVE DIAGNOSIS: Right colon tumor. PO...


In [6]:
df["category"].value_counts()

,count
category,
Orthopedic,223
Gastroenterology,157
Neurology,143
Urology,115


In [7]:
df["text"].sample().values

array([' PROCEDURES: Cystourethroscopy and transurethral resection of prostate. COMPLICATIONS: None. ADMITTING DIAGNOSIS: Difficulty voiding. HISTORY: This 67-year old Hispanic male patient was admitted because of enlarged prostate and symptoms of bladder neck obstruction. Physical examination revealed normal heart and lungs. Abdomen was negative for abnormal findings. LABORATORY DATA: BUN 19 and creatinine 1.1. Blood group was A, Rh positive, Hemoglobin 13, Hematocrit 32.1, Prothrombin time 12.6 seconds, PTT 37.1. Discharge hemoglobin 11.4, and hematocrit 33.3. Chest x-ray calcified old granulomatous disease, otherwise normal. EKG was normal. COURSE IN THE HOSPITAL: The patient had a cysto and TUR of the prostate. Postoperative course was uncomplicated. The pathology report is pending at the time of dictation. He is being discharged in satisfactory condition with a good urinary stream, minimal hematuria, and on Bactrim DS one a day for ten days with a standard postprostatic surgery in

In [8]:
# Prepare text dataframe
text_df = df.copy()

# Create text_id if not exists
if 'text_id' not in text_df.columns:
    text_df['text_id'] = range(len(text_df))

# Ensure text column exists
if 'text' not in text_df.columns:
    # Try to find text column
    text_cols = [col for col in text_df.columns if 'text' in col.lower() or 'description' in col.lower()]
    if text_cols:
        text_df['text'] = text_df[text_cols[0]]
    else:
        raise ValueError("No text column found in dataset")

# Select and rename columns
text_df = text_df[['text_id', 'text']].copy()

# Strip leading/trailing spaces
text_df['text'] = text_df['text'].astype(str).str.strip()

# Save to Google Drive
output_path = Path(f"{PROJECT_FOLDER}/data/processed/text_data.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
text_df.to_csv(output_path, index=False)

print(f"✅ Saved {len(text_df)} texts to {output_path}")
text_df.head()


✅ Saved 638 texts to /content/drive/MyDrive/john_snow_labs_ner/data/processed/text_data.csv


,text_id,text
0,0,PROCEDURES PERFORMED: Colonoscopy. INDICATIONS...
1,1,OPERATION 1. Ivor-Lewis esophagogastrectomy. 2...
2,2,PREOPERATIVE DIAGNOSES: 1. Gastroesophageal re...
3,3,PROCEDURE: Colonoscopy. PREOPERATIVE DIAGNOSES...
4,4,PREOPERATIVE DIAGNOSIS: Right colon tumor. POS...


In [9]:
text_df["text"].sample().values

array(['CC: Progressive loss of color vision OD HX: 58 y/o female presents with a one year history of progressive loss of color vision. In the past two months she has developed blurred vision and a central scotoma OD. There are no symptoms of photopsias, diplopia, headache, or eye pain. There are no other complaints. There have been mild fluctuations of her symptoms, but her vision has never returned to its baseline prior to symptom onset one year ago. EXAM: Visual acuity with correction: 20/25+1 OD; 20/20-1 OS. Pupils were 3.5mm OU. There was a 0.8 log unit RAPD OD. Intraocular pressures were 25 and 24, OD and OS respectively; and there was an increase to 27 on upgaze OD, but no increase on upgaze OS. Optic disk pallor was evident OD, but not OS. Additionally, there was a small area of peripheral chorioretinal scarring in the inferotemporal area of the right eye. Foveal flicker fusion occurred at a frequency of 21.9 OD and 30.7 OS. Color plate testing scores: 6/14 OD and 10/14 OS. Gol

## 4. NER Pipeline Execution


In [10]:
"""
Multi-NER Pipeline with priority + Posology filter
Priority in merged chunks: Posology (only DRUG/DOSAGE) > Clinical > Deid
"""

import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import (
    SentenceDetectorDLModel,
    MedicalNerModel,
    NerConverterInternal,
    ChunkFilterer
)
from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from sparknlp_jsl.annotator.merge.chunk_merge import ChunkMergeApproach



# -------------------------------------------------------
# 1. Base NLP Stages
# -------------------------------------------------------
documentAssembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

sentenceDetector = SentenceDetectorDLModel.pretrained(
    "sentence_detector_dl_healthcare",
    "en",
    "clinical/models"
).setInputCols(["document"]) \
 .setOutputCol("sentence")

tokenizer = Tokenizer() \
    .setInputCols(["sentence"]) \
    .setOutputCol("token")

word_embeddings = WordEmbeddingsModel.pretrained(
    "embeddings_clinical",
    "en",
    "clinical/models"
).setInputCols(["sentence", "token"]) \
 .setOutputCol("embeddings")

# -------------------------------------------------------
# 2. NER Models: Clinical, Deid, Posology
# -------------------------------------------------------
clinical_ner = MedicalNerModel.pretrained(
    "ner_clinical_large",
    "en",
    "clinical/models"
).setInputCols(["sentence", "token", "embeddings"]) \
 .setOutputCol("ner_clinical") \
 .setLabelCasing("upper") #decide if we want to return the tags in upper or lower case

deid_ner = MedicalNerModel.pretrained(
    "ner_deid_generic_augmented",
    "en",
    "clinical/models"
).setInputCols(["sentence", "token", "embeddings"]) \
 .setOutputCol("ner_deid") \
 .setLabelCasing("upper") #decide if we want to return the tags in upper or lower case

posology_ner = MedicalNerModel.pretrained(
    "ner_posology",
    "en",
    "clinical/models"
).setInputCols(["sentence", "token", "embeddings"]) \
 .setOutputCol("ner_posology") \
 .setLabelCasing("upper") #decide if we want to return the tags in upper or lower case

# -------------------------------------------------------
# 3. NerConverterInternal for each model
# -------------------------------------------------------
chunk_clinical = NerConverterInternal() \
    .setInputCols(["sentence", "token", "ner_clinical"]) \
    .setOutputCol("chunk_clinical")

chunk_deid = NerConverterInternal() \
    .setInputCols(["sentence", "token", "ner_deid"]) \
    .setOutputCol("chunk_deid")

chunk_posology = NerConverterInternal() \
    .setInputCols(["sentence", "token", "ner_posology"]) \
    .setOutputCol("chunk_posology")

# -------------------------------------------------------
# 4. Posology filter: only DRUG + DOSAGE
# -------------------------------------------------------
posology_filter = ChunkFilterer() \
    .setInputCols("sentence", "chunk_posology") \
    .setOutputCol("chunk_posology_filtered") \
    .setWhiteList(["DRUG", "DOSAGE"]) \
    .setFilterEntity("entity") \
    .setCriteria("isin") \
    .setCaseSensitive(False)

# Posology ınclude only DRUG , DOSAGE.

# Merging overlapped chunks by considering their lenght
# If we set setOrderingFeatures(["ChunkLength"]) and setSelectionStrategy("DiverseLonger") parameters, the longest chunk will be prioritized in case of overlapping.


# -------------------------------------------------------
# 5. ChunkMergeApproach with priority:
#    Posology (filtered) > Clinical > Deid
# -------------------------------------------------------
chunk_merger = ChunkMergeApproach() \
    .setInputCols("chunk_posology_filtered", "chunk_clinical", "chunk_deid") \
    .setOutputCol("merged_ner_chunk") \
    .setOrderingFeatures(["ChunkLength"]) \
    .setSelectionStrategy("DiverseLonger") \
    .setCaseSensitive(False)

# -------------------------------------------------------
# 6. Full Pipeline
# -------------------------------------------------------
nlpPipeline = Pipeline(stages=[
    documentAssembler,
    sentenceDetector,
    tokenizer,
    word_embeddings,
    # NER models
    clinical_ner,
    deid_ner,
    posology_ner,
    # chunk converters
    chunk_clinical,
    chunk_deid,
    chunk_posology,
    # posology filter (only Drug/Dosage)
    posology_filter,
    # merged chunks
    chunk_merger
])

sentence_detector_dl_healthcare download started this may take some time.
Approximate size to download 367.3 KB
[OK!]
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]
ner_clinical_large download started this may take some time.
Approximate size to download 13.9 MB
[OK!]
ner_deid_generic_augmented download started this may take some time.
Approximate size to download 13.8 MB
[OK!]
ner_posology download started this may take some time.
Approximate size to download 13.8 MB
[OK!]


In [11]:
# Boş model fit
empty_data = spark.createDataFrame([["dummy text"]]).toDF("text")
pipeline_model = nlpPipeline.fit(empty_data)

In [12]:
ner_models = [
    ("clinical_ner", clinical_ner),
    ("deid_ner", deid_ner),
    ("posology_ner", posology_ner)
]


for model_name, ner_model in ner_models:
    print(ner_model.getClasses())

['O', 'B-TREATMENT', 'I-TREATMENT', 'B-PROBLEM', 'I-PROBLEM', 'B-TEST', 'I-TEST']
['O', 'I-LOCATION', 'I-CONTACT', 'I-PROFESSION', 'I-NAME', 'I-DATE', 'B-ID', 'B-CONTACT', 'B-PROFESSION', 'I-ID', 'B-NAME', 'B-DATE', 'B-LOCATION', 'B-AGE', 'I-AGE']
['O', 'B-DOSAGE', 'B-STRENGTH', 'I-STRENGTH', 'B-ROUTE', 'B-FREQUENCY', 'I-FREQUENCY', 'B-DRUG', 'I-DRUG', 'B-FORM', 'I-DOSAGE', 'B-DURATION', 'I-DURATION', 'I-FORM', 'I-ROUTE']


In [13]:
pipeline_model.stages

[DocumentAssembler_0291edfe4454,
 SentenceDetectorDLModel_6bafc4746ea5,
 REGEX_TOKENIZER_b9b65954119b,
 WORD_EMBEDDINGS_MODEL_9004b1d00302,
 MedicalNerModel_1a8637089929,
 MedicalNerModel_e8178a1262cc,
 MedicalNerModel_4a303d875127,
 NER_CONVERTER_a9102b30f883,
 NER_CONVERTER_b1c0892da692,
 NER_CONVERTER_d41104096611,
 ChunkFilterer_509da1d7c261,
 MERGE_e891c4e568c7]

In [14]:
# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(text_df)
print(f"Spark DataFrame created with {spark_df.count()} rows")
spark_df.show(5, truncate=100)

Spark DataFrame created with 638 rows
+-------+----------------------------------------------------------------------------------------------------+
|text_id|                                                                                                text|
+-------+----------------------------------------------------------------------------------------------------+
|      0|PROCEDURES PERFORMED: Colonoscopy. INDICATIONS: Renewed symptoms likely consistent with active fl...|
|      1|OPERATION 1. Ivor-Lewis esophagogastrectomy. 2. Feeding jejunostomy. 3. Placement of two right-si...|
|      2|PREOPERATIVE DIAGNOSES: 1. Gastroesophageal reflux disease. 2. Chronic dyspepsia. POSTOPERATIVE D...|
|      3|PROCEDURE: Colonoscopy. PREOPERATIVE DIAGNOSES: Rectal bleeding and perirectal abscess. POSTOPERA...|
|      4|PREOPERATIVE DIAGNOSIS: Right colon tumor. POSTOPERATIVE DIAGNOSES: 1. Right colon cancer. 2. Asc...|
+-------+-----------------------------------------------------------------

In [15]:
spark_df.printSchema()
spark_df.show(3, truncate=100)


root
 |-- text_id: long (nullable = true)
 |-- text: string (nullable = true)

+-------+----------------------------------------------------------------------------------------------------+
|text_id|                                                                                                text|
+-------+----------------------------------------------------------------------------------------------------+
|      0|PROCEDURES PERFORMED: Colonoscopy. INDICATIONS: Renewed symptoms likely consistent with active fl...|
|      1|OPERATION 1. Ivor-Lewis esophagogastrectomy. 2. Feeding jejunostomy. 3. Placement of two right-si...|
|      2|PREOPERATIVE DIAGNOSES: 1. Gastroesophageal reflux disease. 2. Chronic dyspepsia. POSTOPERATIVE D...|
+-------+----------------------------------------------------------------------------------------------------+
only showing top 3 rows



In [16]:
# Run NER pipeline
print("Running NER pipeline... This may take several minutes...")
result_df = pipeline_model.transform(spark_df)

print("✅ NER pipeline completed")
result_df.select("text_id", "text", "chunk_clinical", "chunk_deid", "chunk_posology").show(5, truncate=100)

Running NER pipeline... This may take several minutes...
✅ NER pipeline completed
+-------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|text_id|                                                                                                text|                                                                                      chunk_clinical|                                                                                          chunk_deid|                                                                                      chunk_posology|
+-------+---------------------------------------------------------------------------------

Spark NLP Annotation bir struct’tur

{
  "result": "Paracetamol",
  "begin": 15,
  "end": 25,
  "metadata": {"entity": "DRUG", ...}
}


In [17]:
result_df.select("merged_ner_chunk").printSchema()


root
 |-- merged_ner_chunk: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- annotatorType: string (nullable = true)
 |    |    |-- begin: integer (nullable = false)
 |    |    |-- end: integer (nullable = false)
 |    |    |-- result: string (nullable = true)
 |    |    |-- metadata: map (nullable = true)
 |    |    |    |-- key: string
 |    |    |    |-- value: string (valueContainsNull = true)
 |    |    |-- embeddings: array (nullable = true)
 |    |    |    |-- element: float (containsNull = false)



## 5. Entity Extraction


In [18]:
from pyspark.sql import functions as F

merged_df = result_df.select(
    "text_id",
    F.explode("merged_ner_chunk").alias("chunk")
).select(
    "text_id",
    F.col("chunk.begin").alias("begin"),
    F.col("chunk.end").alias("end"),
    F.col("chunk.result").alias("chunk"),
    F.col("chunk.metadata")["entity"].alias("entity")
)

entity_df = merged_df.toPandas()
entity_df.head()

,text_id,begin,end,chunk,entity
0,0,22,32,Colonoscopy,TEST
1,0,48,63,Renewed symptoms,PROBLEM
2,0,104,129,Inflammatory Bowel Disease,PROBLEM
3,0,150,169,conventional therapy,TREATMENT
4,0,181,193,sulfasalazine,DRUG


In [19]:
tid = int(entity_df["text_id"].sample(1).iloc[0])
text = text_df.loc[text_df["text_id"] == tid, "text"].values[0]

sample = entity_df[entity_df["text_id"] == tid].head(5)
for _, row in sample.iterrows():
    begin, end, chunk = row["begin"], row["end"], row["chunk"]
    print("TEXT SLICE :", repr(text[begin:end]))
    print("CHUNK      :", repr(chunk))
    print("---")

TEXT SLICE : ''
CHUNK      : 'A'
---
TEXT SLICE : '60-year-ol'
CHUNK      : '60-year-old'
---
TEXT SLICE : 'neuropsychological evaluatio'
CHUNK      : 'neuropsychological evaluation'
---
TEXT SLICE : 'mild cognitive deficit'
CHUNK      : 'mild cognitive deficits'
---
TEXT SLICE : 'a neuropsychological screening evaluatio'
CHUNK      : 'a neuropsychological screening evaluation'
---


In [20]:
entity_output_path = Path(f"{PROJECT_FOLDER}/data/processed/entities.csv")
entity_output_path.parent.mkdir(parents=True, exist_ok=True)

entity_df.to_csv(entity_output_path, index=False, encoding="utf-8")

print(f"\n✅ Saved merged NER entities to: {entity_output_path}")
print(f"Total extracted entities: {len(entity_df)}")
entity_df.head(10)


✅ Saved merged NER entities to: /content/drive/MyDrive/john_snow_labs_ner/data/processed/entities.csv
Total extracted entities: 39865


,text_id,begin,end,chunk,entity
0,0,22,32,Colonoscopy,TEST
1,0,48,63,Renewed symptoms,PROBLEM
2,0,104,129,Inflammatory Bowel Disease,PROBLEM
3,0,150,169,conventional therapy,TREATMENT
4,0,181,193,sulfasalazine,DRUG
5,0,196,204,cortisone,DRUG
6,0,207,219,local therapy,TREATMENT
7,0,272,284,the procedure,TREATMENT
8,0,362,369,bleeding,PROBLEM
9,0,372,380,infection,PROBLEM


In [21]:
entity_df["entity"].value_counts()

,count
entity,
PROBLEM,18259
TREATMENT,11837
TEST,5676
DRUG,1977
DATE,839
AGE,447
LOCATION,275
NAME,257
DOSAGE,214


In [22]:
print(entity_df.dtypes)
print(entity_df[["begin", "end"]].head())
print(entity_df[["begin", "end"]].isna().sum())


text_id     int64
begin       int32
end         int32
chunk      object
entity     object
dtype: object
   begin  end
0     22   32
1     48   63
2    104  129
3    150  169
4    181  193
begin    0
end      0
dtype: int64


In [23]:
# import re

# def fix_entity_offsets_all(text_df, entity_df):
#     corrected = []

#     # text_id bazlı çalış
#     for tid, group in entity_df.groupby("text_id"):
#         text = text_df.loc[text_df["text_id"] == tid, "text"].values[0]

#         for _, row in group.iterrows():
#             begin = int(row["begin"])
#             end   = int(row["end"])
#             chunk = row["chunk"]

#             real = text[begin:end]

#             # 2. Baştaki boşlukları kaldır
#             while real.startswith(" ") and begin < end:
#                 begin += 1
#                 real = text[begin:end]

#             # 3. Eğer chunk real'in devamıysa ama kısa kalmışsa
#             if chunk.startswith(real) and len(real) < len(chunk):
#                 diff = len(chunk) - len(real)
#                 end += diff
#                 real = text[begin:end]

#             # 4. Chunk hâlâ real içinde yoksa, metin içinde ara
#             if chunk not in real:
#                 m = re.search(re.escape(chunk), text)
#                 if m:
#                     begin = m.start()
#                     end   = m.end()

#             corrected.append(
#                 [tid, begin, end, chunk, row["entity"]]
#             )

#     return pd.DataFrame(corrected, columns=entity_df.columns)


In [24]:
# entity_df["entity"].value_counts()

## 6. CoNLL Format Conversion


In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from pyspark.ml import Pipeline
from tqdm import tqdm
from collections import Counter
import pandas as pd
from pathlib import Path
import os


from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from pyspark.ml import Pipeline
from tqdm import tqdm
from collections import Counter
import pandas as pd
from pathlib import Path
import os


def make_conll(
    text: pd.DataFrame,
    entity: pd.DataFrame,
    project_folder: str,
    save_tag: bool = True,
    save_conll: bool = True,
    verbose: bool = None,
    begin_deviation: int = 0,
    end_deviation: int = 0,
) -> str:

    df_text = text.iloc[:, [0, 1]]
#     df_text = (
#     result_df
#     .select("text_id", "text")
#     .limit(5000)
#     .toPandas()
# ) # limit 5k
    print(len(df_text))
    df_entity = entity.iloc[:, [0, 1, 2, 3, 4]]
    df_text.columns = ["text_id", "text"]
    df_entity.columns = ["text_id", "begin", "end", "chunk", "entity"]
    entity_list = list(df_entity.entity.unique())

    ########--------------1.tag transformation function------------########

    def transform_text(text, entities, verbose=None):

        tag_list = []
        for entity in entities.iterrows():

            begin = entity[1][1] + begin_deviation
            end   = int(entity[1][2]) + 1 + end_deviation
            chunk = entity[1][3]
            tag = entity[1][4]
            text = text[:end] + f" </END_NER:{tag}> " + text[end:]
            text = text[:begin] + f" <START_NER:{tag}> " + text[begin:]
            tag_list.append(tag)

        sum_of_added_entity = Counter(tag_list)
        sum_of_entity = Counter(entities["entity"].values)

        if verbose:
            print(f"Processed text id   : {entities.text_id.values[:1]}")
            print(
                f"Original Entities   : {sum_of_entity}\nAdded Entities      : {sum_of_added_entity}"
            )
            print(f"Number Equality     : {sum_of_added_entity == sum_of_entity}")
            print("==" * 40)

        if not sum_of_entity == sum_of_added_entity:
            print("There is a problem in text id:")
            print(entities.text_id.values[0])
            raise Exception("Check this text!")

        return text

    ######---------------2.apply_transform_text function ----------------#######

    def apply_tag_ner(df_text, df_entity, save=None, verbose=None):

        for text_id in tqdm(df_text.text_id):
            text = df_text.loc[df_text["text_id"] == text_id]["text"].values[0]
            entities = df_entity.loc[(df_entity["text_id"] == text_id)].sort_values(
                by="begin", ascending=False
            )

            df_text.loc[df_text["text_id"] == text_id, "text"] = transform_text(
                text, entities, verbose=verbose
            )

        if save:
            df_text.to_csv("text_with_ner_tag.csv", index=False, encoding="utf8")

        return df_text

    ##########----------------3.RUNNING TAG FUNCTION---------------#############

    print("Text tagging starting. Applying entities to whole text...\n")
    df = apply_tag_ner(df_text, df_entity, save=save_tag, verbose=verbose)

    ###########---------------4.Spark Pipeline-----------------------###########

    def spark_pipeline(df):
        spark_df = spark.createDataFrame(df)

        documentAssembler = (
            DocumentAssembler()
            .setInputCol("text")
            .setOutputCol("document")
            .setCleanupMode("shrink")
        )

        sentenceDetector = (
            SentenceDetector()
            .setInputCols(["document"])
            .setOutputCol("sentences")
            .setExplodeSentences(True)
        )

        tokenizer = Tokenizer().setInputCols(["sentences"]).setOutputCol("token")

        nlpPipeline = Pipeline(stages=[documentAssembler, sentenceDetector, tokenizer])

        empty_df = spark.createDataFrame([[""]]).toDF("text")
        pipelineModel = nlpPipeline.fit(empty_df)

        result = pipelineModel.transform(spark_df.select(["text"]))

        return result.select("token.result").toPandas()

    print("\n\nSpark pipeline is running...")

    df_final = spark_pipeline(df)

    #########--------------5.CoNLL Function--------------------#############

    def build_conll(df_final, tag_list, save=None):

        header = "-DOCSTART- -X- -X- O\n\n"
        conll_text = ""
        chunks = []
        tag_list = tag_list
        tag = "O"  # token tag
        ct = "B"  # chunk tag part B or I

        for sentence_tokens in tqdm(df_final.result[:]):
            for token in sentence_tokens:
                if token.startswith("<START_NER:"):
                    tag = token.split(":")[1][:-1]
                    if tag not in tag_list:
                        tag = "O"
                        conll_text += f"{token} NN NN {tag}\n"

                    continue

                if token.startswith("</END_NER:") and tag != "O":
                    for i, chunk in enumerate(chunks):
                        ct = "B" if i == 0 else "I"
                        conll_text += f"{chunk} NNP NNP {ct}-{tag}\n"

                    chunks = []
                    tag = "O"
                    continue

                if tag != "O":
                    chunks.append(token)
                    continue

                if tag == "O":
                    conll_text += f"{token} NN NN {tag}\n"
                    continue

            conll_text += "\n"

        if save:
            output_path = Path(project_folder) / "data/conll/conll2003_text_file.conll"
            output_path.parent.mkdir(parents=True, exist_ok=True)
            with open(output_path, "w+", encoding="utf8") as f:
                f.write(header)
                f.write(conll_text)
            print(f"✅ CoNLL file saved to {output_path}")

        print("\nDONE!")
        return conll_text

    ########----------------6.RUNNING CONLL FUNCTION--------------------########

    print("Conll file is being created...\n")
    return build_conll(df_final, tag_list=entity_list, save=save_conll)



In [26]:
import sys, pkgutil, subprocess
if pkgutil.find_loader("tqdm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])

from tqdm.auto import tqdm
from collections import Counter

In [27]:
conll_text = make_conll(
    text=text_df,
    entity=entity_df,
    project_folder=PROJECT_FOLDER,
    save_tag=True,
    save_conll=True,
    verbose=False
)


638
Text tagging starting. Applying entities to whole text...



  0%|          | 0/638 [00:00<?, ?it/s]



Spark pipeline is running...
Conll file is being created...



  0%|          | 0/25981 [00:00<?, ?it/s]

✅ CoNLL file saved to /content/drive/MyDrive/john_snow_labs_ner/data/conll/conll2003_text_file.conll

DONE!


In [28]:
# Checking conll string
print(conll_text[:570])

PROCEDURES NN NN O
PERFORMED NN NN O
: NN NN O
Colonoscopy NNP NNP B-TEST
. NN NN O

INDICATIONS NN NN O
: NN NN O
Renewed NNP NNP B-PROBLEM
symptoms NNP NNP I-PROBLEM
likely NN NN O
consistent NN NN O
with NN NN O
active NN NN O
flare NN NN O
of NN NN O
Inflammatory NNP NNP B-PROBLEM
Bowel NNP NNP I-PROBLEM
Disease NNP NNP I-PROBLEM
, NN NN O
not NN NN O
responsive NN NN O
to NN NN O
conventional NNP NNP B-TREATMENT
therapy NNP NNP I-TREATMENT
including NN NN O
sulfasalazine NNP NNP B-DRUG
, NN NN O
cortisone NNP NNP B-DRUG
, NN NN O
local NNP NNP B-TREATMENT
the


## 7. CoNLL File Validation

Verify that the CoNLL file was created correctly and can be read by Spark NLP

In [29]:
# Read and display sample CoNLL content
conll_file_path = Path(f"{PROJECT_FOLDER}/data/conll/conll2003_text_file.conll")

print("📄 CoNLL File Sample (First 100 lines):")
print("=" * 80)

with open(conll_file_path, 'r', encoding='utf8') as f:
    lines = f.readlines()
    for i, line in enumerate(lines[:100], 1):
        print(f"{i:3d}: {line}", end='')

print("\n" + "=" * 80)
print(f"\n📊 CoNLL File Statistics:")
print(f"  Total lines: {len(lines)}")
print(f"  File size: {conll_file_path.stat().st_size / 1024:.2f} KB")
print(f"  File location: {conll_file_path}")

📄 CoNLL File Sample (First 100 lines):
  1: -DOCSTART- -X- -X- O
  2: 
  3: PROCEDURES NN NN O
  4: PERFORMED NN NN O
  5: : NN NN O
  6: Colonoscopy NNP NNP B-TEST
  7: . NN NN O
  8: 
  9: INDICATIONS NN NN O
 10: : NN NN O
 11: Renewed NNP NNP B-PROBLEM
 12: symptoms NNP NNP I-PROBLEM
 13: likely NN NN O
 14: consistent NN NN O
 15: with NN NN O
 16: active NN NN O
 17: flare NN NN O
 18: of NN NN O
 19: Inflammatory NNP NNP B-PROBLEM
 20: Bowel NNP NNP I-PROBLEM
 21: Disease NNP NNP I-PROBLEM
 22: , NN NN O
 23: not NN NN O
 24: responsive NN NN O
 25: to NN NN O
 26: conventional NNP NNP B-TREATMENT
 27: therapy NNP NNP I-TREATMENT
 28: including NN NN O
 29: sulfasalazine NNP NNP B-DRUG
 30: , NN NN O
 31: cortisone NNP NNP B-DRUG
 32: , NN NN O
 33: local NNP NNP B-TREATMENT
 34: therapy NNP NNP I-TREATMENT
 35: . NN NN O
 36: 
 37: PROCEDURE NN NN O
 38: : NN NN O
 39: Informed NN NN O
 40: consent NN NN O
 41: was NN NN O
 42: obtained NN NN O
 43: prior NN NN O
 44: to NN NN 

## Summary

✅ **Data preparation completed!**

**Created files in Google Drive:**
- `data/processed/text_data.csv` - Prepared text data
- `data/processed/entities.csv` - Extracted entities
- `data/conll/conll2003_text_file.conll` - CoNLL format training data

**Next step:** Run `training.ipynb` notebook to train the custom NER model.
